In [8]:
import site
site.getsitepackages()

['c:\\Users\\Francisco Azeredo\\.conda\\envs\\tese',
 'c:\\Users\\Francisco Azeredo\\.conda\\envs\\tese\\Lib\\site-packages']

In [9]:
import weaviate

In [10]:
client = weaviate.connect_to_local()

In [11]:
response = client.collections.list_all(simple=False)

print(response)

{'DemoCollection': _CollectionConfig(name='DemoCollection', description=None, generative_config=_GenerativeConfig(generative=<GenerativeSearches.OLLAMA: 'generative-ollama'>, model={'apiEndpoint': 'http://host.docker.internal:11434', 'model': 'llama3.2'}), inverted_index_config=_InvertedIndexConfig(bm25=_BM25Config(b=0.75, k1=1.2), cleanup_interval_seconds=60, index_null_state=False, index_property_length=False, index_timestamps=False, stopwords=_StopwordsConfig(preset=<StopwordsPreset.EN: 'en'>, additions=None, removals=None)), multi_tenancy_config=_MultiTenancyConfig(enabled=False, auto_tenant_creation=False, auto_tenant_activation=False), properties=[_Property(name='page', description=None, data_type=<DataType.INT: 'int'>, index_filterable=True, index_range_filters=False, index_searchable=False, nested_properties=None, tokenization=None, vectorizer_config=None, vectorizer='none'), _Property(name='text', description=None, data_type=<DataType.TEXT: 'text'>, index_filterable=True, inde

In [12]:
client.collections.delete("DemoCollection")  # THIS WILL DELETE THE SPECIFIED COLLECTION(S) AND THEIR OBJECTS

Collection creation

In [13]:
from weaviate.classes.config import Configure, Property, DataType

client.collections.create(
    "DemoCollection",
    properties=[
        Property(name="page", data_type=DataType.INT),
        Property(name="text", data_type=DataType.TEXT)
    ],
    # vectorizer_config=Configure.Vectorizer.text2vec_ollama(),
    vectorizer_config=[
        Configure.NamedVectors.text2vec_ollama(
            name="title_vector",
            source_properties=["page", "text"],
            api_endpoint="http://host.docker.internal:11434",
            model="mxbai-embed-large"
        )
    ],
    generative_config=Configure.Generative.ollama(
        api_endpoint="http://host.docker.internal:11434",  # If using Docker, use this to contact your local Ollama instance
        model="llama3.2"  # The model to use, e.g. "phi3", or "mistral", "command-r-plus", "gemma"
    )
    # Additional parameters not shown
)

Data Import

In [ ]:
source_objects = {  
  "Resumo": [
    "A extraccao de palavras chaves e uma tarefa importante no processamento de lingua natural.",
    "Palavras chaves podem, por exemplo, facilitar o processo de resumir uma coleccao de documentos ao descreverem cada documento com concisao.",
    "Adicionalmente, estas facilitam o processo de visualizacao de padroes que existem em documentos textuais, alem de facilitar outras tarefas, como categorizacao, agrupamento, indexacao e pesquisa."
  ],
  "Metodos_de_extraccao": [
    {
      "Nome": "Abordagem supervisada",
      "Descrição": "depende de dados de treino"
    },
    {
      "Nome": "Método não supervisionado",
      "Descrição": "tenta inferir a relevancia de palavras chave a partir de estatisticas directamente extraidas de um conjunto de dados"
    }
  ],
  "Novo_metodo": [
    {
      "Nome": "Abordagem baseada em medidas de centralidade",
      "Descrição": "combinando uma abordagem com medidas de centralidade sobre um grafo ponderado"
    },
    {
      "Nome": "Técnicas de modelos linguísticos",
      "Descrição": "inovadora de diferentes abordagens para estimar a importancia de uma palavra chave e o grau da relacao semantica entre candidatos a palavras chave"
    }
  ],
  "Tecnica_de_ajuste": [
    {
      "Nome": "Estimativa do grau de autocorrelacao espacial",
      "Descrição": "com o objetivo da captura da palavras chave candidata com padroes de distribuicao espacial interessantes"
    }
  ],
  "Testes": [
    {
      "Nome": "Corpora diferentes",
      "Descrição": "três corpora diferentes frequentemente usados para avaliar os metodos de extraccao de palavras chave"
    }
  ],
  "Resultados": [
    {
      "Nome": "Avaliacao",
      "Descrição": "os resultados obtidos com o novo metodo aproximam e, em alguns casos, superam os resultados obtidos por outros metodos estado-da-arte para extraccao de palavras chave"
    }
  ]
}

In [ ]:
collection = client.collections.get("DemoCollection")

with open("output.txt", "r", encoding="utf-8") as f:
    page_texts = f.read().strip().split("---")

# ✅ Open batch context once (outside the loop)
with collection.batch.dynamic() as batch:
    for idx, text in enumerate(page_texts):
        batch.add_object(
            properties={"page": idx + 1, "text": text.strip()}  # ✅ Correct parameter name
        )
        batch.flush()  # ✅ Flush after each object


In [ ]:
print(collection.batch.failed_objects)
result = collection.query.fetch_objects(limit=10)  # Adjust limit as needed

for obj in result.objects:
    print(f"Page {obj.properties['page']}: {obj.properties['text'][:200]}...")  # Show first 200 chars


NameError: name 'collection' is not defined

Query by text search

In [ ]:
query_text = "keyphrase extraction"

result = collection.query.near_text(
    query=query_text,  # ✅ Use query inside near_text
    limit=5  # ✅ Limit to 5 results
)

if result.objects:
    for obj in result.objects:
        print(f"Page {obj.properties['page']}: {obj.properties['text']}...")
else:
    print("No relevant results found.")

Page 13: Page 13:


###### Contents

* 1 Introduction
	* 1.1 Objectives
	* 1.2 Methodology
	* 1.3 Results and Contributions
	* 1.4 Outline of the Document
* 2 Concepts and Related Work
	* 2.1 Fundamental Concepts
		* 2.1.1 Sequence Tagging for Natural Language Processing
		* 2.1.2 Part-of-Speech Tagging
		* 2.1.3 Noun Phrase Chunking
	* 2.2 Related Work on Unsupervised Keyphrase Extraction
		* 2.2.1 Graph-Based Techniques for Keyphrase Extraction
		* 2.2.2 Language Modeling Techniques for Keyphrase Extraction
	* 2.3 Overview
* 3 A Hybrid Method for Keyphrase Extraction
	* 3.1 Candidate Selection
	* 3.2 Initial Ranking
	* 3.3 Re-Ranking
	* 3.4 Summary
* 4...
Page 9: Page 9:


###### Abstract

Keyphrase extraction is an important task in natural language processing. Keyphrases can, for instance, ease the process of summarizing a collection of documents by concisely describing each document. Furthermore, they facilitate the process of visualizing patterns that exist in textual documents, 

In [ ]:
client.close()